# 014 — True-label threshold calibration and probability ensembles

Модели 009–013 обучены только на слабой разметке вне публичных пяти минут. Здесь веса моделей не изменяются. Первые четыре публичные минуты используются ТОЛЬКО для выбора модели/комбинации, порога и длительности D; последняя минута — ранее исследованная диагностика, не независимый тест. Результат на всех пяти минутах включает подбор и не является held-out.

Предварительно запустить 010–013. Общие функции предобработки и предсказания читаются из ячейки phase-0 ноутбука 010 (обучающая ячейка не исполняется). При отсутствии локальных вероятностей они восстанавливаются из опубликованных моделей. Признаки и маски — локальные зависимости 009.

Кандидаты: каждый из пяти бустингов/логрегрессии/лесов/MLP по отдельности, среднее, минимум, геометрическое среднее и вторая по величине вероятность (согласие хотя бы двух). Перебор тех же порогов и длительностей, минимум 100 предсказанных кадров D при подборе. Для каждой целевой precision .95/.99 выбирается максимальный recall. Пороги фиксируются ДО чтения результатов последней минуты. Это проверка переноса высокой точности, а не обещание precision.

Публикуются только код и конфигурация. Предсказания остаются локальными. Бинарная задача D/LIVE, без spellF1 и трёхклассового macroF1. Новых обучаемых весов у ансамбля нет; используются *_weak_model.pkl из родительских решений и model.pkl из 009.


In [ ]:
from pathlib import Path
import json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'games.py').exists())
EXPERIMENT='014_dead_precision_calibration'
shared=json.loads((ROOT/'sol/010_dead_logistic/experiment.ipynb').read_text())
exec(''.join(next(c for c in shared['cells'] if c.get('id')=='phase-0')['source']))
parents=['009_broadcast_weak_boosting','010_dead_logistic','011_dead_forests','012_dead_boost_sweep','013_dead_mlp']
probabilities=[]
for parent in parents:
 local=ROOT/'local_review/experiments'/parent
 path=local/('p_D.npy' if parent.startswith('009') else 'weak_p_D.npy')
 if path.exists():p=np.load(path)
 else:
  artifact=ROOT/'sol'/parent/('model.pkl' if parent.startswith('009') else 'weak_model.pkl')
  with artifact.open('rb') as f:model=pickle.load(f)
  if parent.startswith('009'):model=dict(kind='boost',model=model)
  p=predict(model,np.arange(N))
 probabilities.append(p)
stack=np.stack(probabilities)
combined={parent[:3]:p for parent,p in zip(parents,probabilities)}
combined.update(mean=stack.mean(axis=0),minimum=stack.min(axis=0),
 geometric=np.exp(np.log(np.clip(stack,1e-7,1)).mean(axis=0)),
 second_highest=np.sort(stack,axis=0)[-2])
val_ids=np.arange(START,START+6000)
choices_by_model={};search_rows=[]
for name,p in combined.items():
 # Full offline prediction context, but ONLY prefix labels enter selection.
 # Полный контекст предсказаний, но в подбор входят ТОЛЬКО метки префикса.
 mask=(positions>=START)&(positions<START+6000)
 choices,rows=choose_settings(np.where(graded,p,-1.),truth,mask)
 choices_by_model[name]=choices
 for row in rows:search_rows.append(dict(variant=name,**row))
# Select without looking at the last-minute scores.
# Выбираем без просмотра результатов последней минуты.
selected={}
for target in TARGETS:
 name=max(combined,key=lambda name:(choices_by_model[name][str(target)]['feasible'],choices_by_model[name][str(target)]['D_recall'],choices_by_model[name][str(target)]['D_precision']))
 selected[str(target)]=dict(model=name,**choices_by_model[name][str(target)])
save_json(OUT/'selected_before_test.json',selected)
results={};all_candidates=[]
for name,p in combined.items():
 for target in TARGETS:
  choice=choices_by_model[name][str(target)]
  pred=np.zeros(N,bool)
  for _,a,b in BLOCKS:pred[a:b+1]=filter_segment(p[a:b+1]>=choice['threshold'],choice['min_dead_frames'])
  metrics={window:score(truth[mask],pred[mask]) for window,mask in [('public_5min',public),('last_minute',public&(positions>=START+6000))]}
  all_candidates.append(dict(model=name,target=target,selection=choice,metrics=metrics))
  if selected[str(target)]['model']==name:
   tag=f'calibrated_p{int(target*100)}'
   labels=np.where(graded,np.where(pred,'D','LIVE'),'X')
   pd.DataFrame({'frame':np.arange(N),'label':labels}).to_csv(OUT/f'{tag}_predictions.csv',index=False)
   np.save(OUT/f'{tag}_dead_mask.npy',pred);np.save(OUT/f'{tag}_p_D.npy',p)
   results[str(target)]=dict(selection=selected[str(target)],metrics=metrics,prediction_file=f'{tag}_predictions.csv')
save_json(OUT/'results.json',dict(outputs=results,candidates=all_candidates,selection_window='public first four minutes',
 evaluation_context='public last minute diagnostic; offline filter; no public labels in underlying model fits'))
pd.DataFrame(search_rows).to_csv(OUT/'development_threshold_search.csv',index=False)
save_json(OUT/'configuration.json',dict(parents=parents,selected=selected,thresholds=THRESHOLDS,minimum_durations=MIN_DURATIONS,
 selection_range=[START,START+6000],diagnostic_range=[START+6000,END],feature_names=feature_names))
progress(stage='completed')
print(json.dumps(results,indent=2))
